In [ ]:
# Testkode: 
import numpy as np
import pandas as pd

print("Python fungerer!")
print("NumPy-versjon:", np.__version__)
print("Pandas-versjon:", pd.__version__)

# Task 1: File ingestion and regular expressions

## 1. Robust File Reader Function
The goal: 
Create a function that reads a file and
returns both its contents and the character encoding that worked.

In [ ]:
from pathlib import Path

# Write the function
def read_file(file_path):
    """
    Reads a file using UTF-8 encoding, falling back to ISO-8859-1
    if UTF-8 decoding fails.

    Args:
        file_path: The path to the file to read.

    Returns:
        A tuple containing the file contents as a string and the
        encoding used ("UTF-8" or "ISO-8859-1").
    """

    file_path = Path(file_path)  # Convert the file path to a Path object

    try:
        text = file_path.read_text(encoding="utf-8") # reads the file with UTF-8
        return text, "UTF-8"                         # returns the text and encoding

    except UnicodeDecodeError:                            # if UTF-8 decoding fails, it falls back to ISO-8859-1
        text = file_path.read_text(encoding="ISO-8859-1") # reads the file with ISO-8859-1
        return text, "ISO-8859-1"                         # returns the text and encoding

    

## 2. Log Parser Function
The goal: 
Create a function that reads a log file, 
identifies valid sales transactions using regular expressions (re), 
returns the information structured as a list of dictionaries.

Valid line format:
[2026-10-15 08:30:00] [POS_01] item=Latte price=4.50 EUR (SUCCESS)

Regular expression (Regex): 
The pattern used to find and extract the six fields from lines like these

In [ ]:
import re 

# Write the function to parse the log file
def parse_log_file(file_text_content):
    """ 
    Parses the log file content and extracts transaction information.

    Args:
        file_text_content: The content of the log file as a string.

    Returns:
        A list of dictionaries, each containing information about a transaction.
    """

# Making the regex-pattern
    pattern = (    
        r"\[(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\] "
        r"\[(POS_\d+)\] "
        r"item=(.*?) "
        r"price=(\d+\.\d{2}) "
        r"(EUR|NOK|USD) "
        r"\((SUCCESS|REFUND|CANCELLED)\)"
    )

    transactions = []

    for line in file_text_content.splitlines():
        match = re.search(pattern, line)        # searches for matches in the line using the regex pattern

        if match:
            timestamp, register_id, item, price, currency, status = match.groups() # extracts the matched groups from the regex match

        # Create dictionary (for transaction)
            transaction = {
                "timestamp": timestamp,
                "register_id": register_id,
                "item": item,
                "price": float(price), 
                "currency": currency,
                "status": status
            }

            transactions.append(transaction)  # adds the new dictionary to the transactions list.

    return transactions # When all lines have been examined, --> function returns the list containing all the transactions it found.

# Task 2: Recursive utility functions

## 1. Recursive Directory Search Function

The goal: 
The function finds all log files in a nested directory
and returns a list of pathlib.Path objects

So basically: 
--> Create a single function that finds all .log files in a directory and all its subdirectories using recursion.

In [ ]:
from pathlib import Path

# Write the function to find log files in a directory
def find_log_files(directory):
    """
    Recursively searches for .log files in the specified directory and its subdirectories.
    
    Args:
        directory: The path to the directory to search.
    
    Returns:
        A list of Path objects representing the found .log files.
    """
    directory = Path(directory)
    log_files = []

    for item in directory.iterdir():
        # Recursion: If the item is a directory, call find_log_files on it; if it's a .log file, add it to the list
        if item.is_dir():
            log_files.extend(find_log_files(item))
        elif item.is_file() and item.suffix == ".log":
            log_files.append(item)

    return log_files

## 2. New Recursive Function
Function: search for active registers in the nested structure of store_config.json.

In [ ]:
# Write the function to find active registers in the parsed data
def find_active_registers(data):
    """
    Finds all active registers in the parsed log data.

    Args:
        data: The parsed log data (dict or list).

    Returns:
        A set of strings representing the IDs of active registers.
    """

    # A set stores unique values. set[str] means that the set should contain strings, such as "POS_01"
    active_registers: set[str] = set()

    if isinstance(data, dict): # Checks if data is a dictionary
        if data.get("active") is True and isinstance(data.get("id"), str):  # Checks if the "active" key is True and the "id" key is a string
            active_registers.add(data["id"])                                # adds the value of the "id" key to the set of active registers

        for value in data.values(): # Examine all the values in the dictionary
            active_registers.update(find_active_registers(value))

    elif isinstance(data, list): # Checks if data is a list: If the data is a list, the function iterates through each element and examines it in the same way
        for item in data:
            active_registers.update(find_active_registers(item))

    return active_registers

# Task 3: Functions for Cleaning and Analysing Data using NumPy

## 1. Function for Removing Price Outliers 
The function should perform four tasks:
1. Converts the input prices into a 1D NumPy array of floats.
2. Finds outliers that are more than 3 standard deviations from the mean.
3. Replaces outliers with the median of the valid prices.
4. Returns the cleaned array. 

In [ ]:
import numpy as np

# Write the function to clean price outliers
def clean_price_outliers(prices):
    """Cleans price outliers from a list of prices.
    
    Args:
        prices: A list or array of prices (floats or ints).

    Returns:
        A numpy array of prices with outliers replaced by the median price.
    """
    prices = np.asarray(prices, dtype=float)  # Convert the input into a NumPy array, dtype=float ensures that the values are floats

# Calculate the mean, standard deviation, and median of the prices
    mean = np.mean(prices)
    std = np.std(prices)
    median = np.median(prices)

# Find outliers: prices that are more than 3 standard deviations away from the mean
    outliers = np.abs(prices - mean) > 3 * std

# Replaces outliers with the median
    prices[outliers] = median

# Returns the cleaned 1D array
    return prices

## 2. Function: Calculates the Proportion of Sales that are Refunds

Function: compute_refund_ratio(sales_count, refund_count)

--> Divide the number of refunds by the number of sales, element by element, and add \(10^{-6}\) to the denominator to avoid division by zero.

Formula:  Refund Ratio = Refund Count/Sales Count + 10^-6


In [ ]:
# Write function to compute refund ratio
def compute_refund_ratio(
    sales_count: np.ndarray,
    refund_count: np.ndarray
) -> np.ndarray:
    """
    Computes the refund ratio for each register.

    Args:
        sales_count: A numpy array containing the number of sales for each register.
        refund_count: A numpy array containing the number of refunds for each register. 

    Returns:
        A numpy array containing the refund ratio for each register.
    """
    
    epsilon = 10**-6   # Saves the value of 10^-6 in a variable called epsilon. 

    refund_ratio = refund_count / (sales_count + epsilon) # formula for calculating the refund ratio, adding epsilon to avoid division by zero

    return refund_ratio  # the result is returned as a numpy array containing the refund ratio for each register.

# Task 4: Data analysis using Pandas 
Transform transaction records into a structured Pandas DataFrame and perform an analyses 

### 4.1 create a DataFrame to sort the transactions from the log files

In [ ]:
from pathlib import Path

print("Working directory:", Path.cwd())
print("Data location:", Path("data").resolve())

log_files = find_log_files("data")
print("Log files found:", len(log_files))

transactions = []

for file in log_files:
    file_content, encoding = read_file(file)
    parsed = parse_log_file(file_content)
    transactions.extend(parsed)

print("Transactions parsed:", len(transactions))

In [ ]:
import pandas as pd
import numpy as np

transactions = []

# Read and parse all log files in the "data" directory
for file in find_log_files("data"):
    file_content, encoding = read_file(file)
    transactions.extend(parse_log_file(file_content))


# function to create and organize the Dataframe
def create_dataframe(transactions: list[dict]) -> pd.DataFrame:
    #convert the list of dictionaries into a pandas DataFrame
    df = pd.DataFrame(transactions)

    #convert the timestamp column to datetime format
    df['timestamp'] = pd.to_datetime(df['timestamp'])

    #set the timestamp column as the index of the DataFrame
    df.set_index('timestamp', inplace=True)

    #sort the DataFrame chronologically by timestamp
    df = df.sort_index()

    return df

# Create the DataFrame and display its information
df = create_dataframe(transactions)

# check the number of transactions and the type of the index
print(f"number of transactions: {len(df)}")
print(f"index type: {type(df.index)}")

display(df.head())  


### 4.2 group data by items and resample

In [ ]:
def hourly_revenue(df: pd.DataFrame) -> pd.DataFrame:
    """Calculate total sales for each item in 1-hour intervals."""

    #group the DataFrame by the "item" column
    grouped_df = df.groupby('item')

    #resample transactions in 1-hour intervals and calculate the total sales for each item
    grouped_df = grouped_df.resample('1H').agg({'price': 'sum'})

    return grouped_df


### 4.3 create a pivot table displaying total transaction revenue

In [ ]:
#create a pivot table to calculate total sales for each item across status categories
def status_revenue_pivot(df: pd.DataFrame) -> pd.DataFrame:
    """Calculate total sales for each item category."""

    #creates the pivot table
    pivot_df = pd.pivot_table(
        df,
        values='price',
        index='item',
        columns='status',
        aggfunc='sum',
        fill_value=0
    )

    return pivot_df

pivot_df = status_revenue_pivot(df)

display(pivot_df)  # Display the pivot table showing total sales for each item across status categories

### Taks 4.4: Calculating refund ratios for each item

create a function that calculates the refund ratio, we also include the refund count and sales count for better interpretation 

In [ ]:
def refund_ratio_per_item(df: pd.DataFrame) -> pd.DataFrame:
    """Calculate the refund ratio for each item."""

    #group the DataFrame by the "item" column
    grouped_df = df.groupby('item')

    #calculate the number of sales and refunds for each item
    sales_count = grouped_df['status'].apply(lambda x: (x == 'SUCCESS').sum())
    refund_count = grouped_df['status'].apply(lambda x: (x == 'REFUND').sum())

    #compute the refund ratio using the compute_refund_ratio function
    refund_ratio = compute_refund_ratio(sales_count.to_numpy(), refund_count.to_numpy())

    #create a new DataFrame to store the results
    result_df = pd.DataFrame({
        'sales_count': sales_count,
        'refund_count': refund_count,
        'refund_ratio': refund_ratio
    })

    return result_df


refund_df = refund_ratio_per_item(df)
display(refund_df)



# Task 5: Creating the full workflow

In [ ]:
from pathlib import Path
import json

def run_workflow(data_folder: str | Path = "data") -> pd.DataFrame:
    """
    Runs the entire workflow: reads log files, parses them, creates a DataFrame,
    calculates hourly revenue, status revenue pivot, and refund ratios.

    Args:
        data_folder: The folder containing the log files.

    Returns:   
        A DataFrame containing cleaned and corrected transaction written to csv files

    """

    data_folder = Path(data_folder)

    #find log files and active registers 
    log_files = find_log_files(data_folder)

    with open(data_folder / "store_config.json", "r", encoding="utf-8") as f:
        config = json.load(f)

    active_registers = find_active_registers(config)


    transactions = []

    #read and parse transactions
    for file in log_files:
        file_content, encoding = read_file(file)
        transactions.extend(parse_log_file(file_content))

    df = create_dataframe(transactions)

    #replace original prices with cleaned values
    df["price"] = clean_price_outliers(df["price"].to_numpy())

    #Correct NOK prices
    df.loc[df["currency"] == "NOK", "price"] *= 10.0

    #Generate reports
    hourly_df = hourly_revenue(df)
    pivot_df = status_revenue_pivot(df)
    refund_df = refund_ratio_per_item(df)

    output_dir = data_folder.parent

    #Export output files
    df.reset_index().to_csv("cleaned_transactions.csv", index=False)

    hourly_df.reset_index().to_json(
        "hourly_revenue.json",
        orient="records",
        date_format="iso",
        indent=2
    )

    pivot_df.to_csv("status_pivot_table.csv")

    refund_df.to_csv("item_refund_ratios.csv")

    return df


df = run_workflow("data")
display(df.head())
    
 

KeyError: 'timestamp'